## Training

In [ ]:
import torch

### Normal Backward Code

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)
cpu_input = torch.randn(128, 128).to(device).requires_grad_()
cpu_weight = torch.randn(128, 128).to(device).requires_grad_()
cpu_target = torch.randn(128, 128).to(device).requires_grad_()

opt_fn = torch.compile(torch.matmul)
cpu_output = opt_fn(cpu_input, cpu_weight)

loss_fn = torch.nn.CrossEntropyLoss()
cpu_loss = loss_fn(cpu_output, cpu_target)
cpu_loss.backward()

### PyTorchSim Backward Code

In [ ]:
device = torch.device("npu:0")

torch.manual_seed(0)
npu_input = torch.randn(128, 128).to(device).requires_grad_()
npu_weight = torch.randn(128, 128).to(device).requires_grad_()
npu_target = torch.randn(128, 128).to(device).requires_grad_()

opt_fn = torch.compile(torch.matmul)
npu_output = opt_fn(npu_input, npu_weight)

loss_fn = torch.nn.CrossEntropyLoss()
npu_loss = loss_fn(npu_output, npu_target)
npu_loss.backward()

In [ ]:
def test_result(name, npu_output, cpu_output, rtol=1e-4, atol=1e-4, preview=5):
    npu_output_cpu = npu_output.cpu()

    if torch.allclose(npu_output_cpu, cpu_output, rtol=rtol, atol=atol):
        message = f"|{name} Functionality Test Passed|"
        print("-" * len(message))
        print(message)
        print("-" * len(message))

        npu_flat = npu_output_cpu.flatten()
        cpu_flat = cpu_output.flatten()

        npu_preview = ", ".join(f"{x.item():.3f}" for x in npu_flat[:preview])
        cpu_preview = ", ".join(f"{x.item():.3f}" for x in cpu_flat[:preview])

        print(f"npu output: [{npu_preview}, ...]")
        print(f"cpu output: [{cpu_preview}, ...]")

    else:
        message = f"|{name} Functionality Test Failed|"
        print("-" * len(message))
        print(message)
        print("-" * len(message))
        print("npu output:", npu_output_cpu)
        print("cpu output:", cpu_output)
        exit(1)

In [ ]:
test_result("MatMul Input Grad", npu_input.grad, cpu_input.grad)
test_result("MatMul Weight Grad", npu_weight.grad, cpu_weight.grad)